In [ ]:
import teehr
from teehr.evaluation.spark_session_utils import create_spark_session

import os
from pathlib import Path
from pyspark.sql import functions as F

In [ ]:
%%time
spark = create_spark_session(
    start_spark_cluster=True,
    executor_instances=64,
    executor_memory="32g",
    executor_cores=4,
    update_configs={
        "spark.sql.shuffle.partitions": "512",  # 6.2B ÷ 500 = 12.4M rows/partition
        # "spark.sql.adaptive.enabled": "true",  # These are set by default
        # "spark.sql.adaptive.coalescePartitions.enabled": "true",
    },
    aws_profile="admin-user"
)

In [ ]:
ev = teehr.RemoteReadWriteEvaluation(spark=spark)

In [ ]:
# cache_dir = '/data/data_processing/backfill-nwm/nwm-medium-teehr-warehouse/local/cache/fetching/nwm/nwm30_medium_range/streamflow_hourly_inst'
# cache_dir = "/data/data_processing/backfill-nwm/nwm-medium-teehr-warehouse/local/cache/fetching/nwm/nwm30_medium_range_blend/streamflow_hourly_inst"  # Medium blend CONUS
# cache_dir = "/data/data_processing/backfill-nwm/nwm-alaska-teehr-warehouse/local/cache/fetching/nwm/nwm30_medium_range_blend_alaska/streamflow_hourly_inst" # Medium blend Alaska
cache_dir = "/data/data_processing/backfill-nwm/nwm-alaska-teehr-warehouse/local/cache/fetching/nwm/nwm30_medium_range_alaska/streamflow_hourly_inst" # Medium Alaska

In [ ]:
files = list(Path(cache_dir).glob("**/*.parquet"))
total_size = sum(f.stat().st_size for f in files)
print(f"Files: {len(files)}")
print(f"Total size: {total_size / (1024**3):.2f} GB")

In [ ]:
nwm_sdf = ev.spark.read.parquet(cache_dir)
nwm_sdf.count()

In [ ]:
nwm_sdf.select(F.min("reference_time")).show(), nwm_sdf.select(F.max("reference_time")).show()

In [ ]:
nwm_sdf.show(n=5, truncate=False)

#### Load to the warehouse

Add the configuration_name if not already present

In [ ]:
ev.configurations.to_sdf().select("name", "description").show(truncate=False)

In [ ]:
ev.configurations.add(
    configuration=[
        teehr.Configuration(
            name="nwm30_medium_range_alaska",
            timeseries_type="secondary",
            description="Alaska NWM medium range, GFS forcing",
        )
    ]
)

This took ~30 mins to write the CONUS medium blend data

In [ ]:
%%time
ev.secondary_timeseries.load_dataframe(nwm_sdf)

#### OLD NOTES: This was run when loading the ~2 year data set

In [ ]:
validated_checkpoint_path = "/data/playground/slamont/teehr/warehouse/loading/backfill_nwm/local-evaluation-medium/local/cache/fetching/nwm/nwm30_medium_range/validated_checkpoint"

In [ ]:
%%time
validated_sdf = ev._validate.dataframe(
    nwm_sdf,
    table_schema=ev.secondary_timeseries.schema_func(),
    uniqueness_fields=ev.secondary_timeseries.uniqueness_fields,
    foreign_keys=ev.secondary_timeseries.foreign_keys
)
validated_sdf.write.parquet(validated_checkpoint_path)


In [ ]:
validated_sdf = spark.read.parquet(validated_checkpoint_path)

In [ ]:
final_sdf = validated_sdf.repartitionByRange(
    512,                                          # Target count
    F.date_trunc("month", F.col("value_time")),   # Primary: month buckets
    F.col("value_time")                           # Secondary: fine-grained time order
)

In [ ]:
%%time
ev._write.to_warehouse(
    final_sdf,
    table_name="secondary_timeseries",
    write_mode="append",
    uniqueness_fields=ev.secondary_timeseries.uniqueness_fields,
)

In [ ]:
sdf = ev.secondary_timeseries.filter([
    "configuration_name = 'nwm30_medium_range'"
]).to_sdf()

In [ ]:
sdf.count()

In [ ]:
sdf.selectExpr("min(value_time)").show()

In [ ]:
spark.stop()